In [ ]:
import os
import time
import warnings
import requests
from io import StringIO

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

# Statistics
from scipy.spatial.distance import pdist
from scipy.stats import pearsonr, entropy
import ptitprince as pt

# BioPython (Updated for modern versions)
from Bio.PDB import MMCIFParser, Superimposer
from Bio.PDB.Polypeptide import three_to_one
from Bio import pairwise2
from Bio.Align import substitution_matrices

warnings.filterwarnings("ignore")

# ==============================================================================
# 0. CONFIGURATION
# ==============================================================================
# Adjust these paths relative to where the notebook is running (analysis/)
DATA_DIR = "../data/"
CIF_DIR = os.path.join(DATA_DIR, "CIF_Files/") 

# Input Files (Processed Data)
# Ensure these exist in your data directory
MSA_FILE = os.path.join(DATA_DIR, 'resources/MSA_DF.csv')
REP_CHAIN_FILE = os.path.join(DATA_DIR, 'resources/Rep_GPCR_chain.csv')
REP_APO_FILE = os.path.join(DATA_DIR, 'resources/Representative_Apo_Structures.csv')
SEQUENCE_INFO_FILE = os.path.join(DATA_DIR, 'resources/Human_GPCR_PDB_Info.csv')
CLASSIFICATION_FILE = os.path.join(DATA_DIR, 'resources/GPCR_PDB_classification.csv')

# Output Directory
OUTPUT_DIR = "./results/correlation_analysis/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Output Files
TM_CORR_RESULTS = os.path.join(OUTPUT_DIR, "MSA_vs_3D_Correlation_TM.csv")
DISP_RESULTS = os.path.join(OUTPUT_DIR, "GPCR_Residue_Displacements.csv")
CONSV_RESULTS = os.path.join(OUTPUT_DIR, "GPCR_MSA_Conservation.csv")
FINAL_CORR_RESULTS = os.path.join(OUTPUT_DIR, "Correlation_Per_GPCR_Dynamics.csv")

print("Configuration loaded. Ready for analysis.")

In [ ]:
# ==============================================================================
# 1. HELPER FUNCTIONS
# ==============================================================================

def get_tm_boundaries(entry_name):
    """Fetches TM residue boundaries from GPCRdb API."""
    if not entry_name: return None
    api_url = f"https://gpcrdb.org/services/residues/{entry_name.lower()}/"
    try:
        response = requests.get(api_url, timeout=20)
        response.raise_for_status()
        data = response.json()
        tm_residues = {res['sequence_number'] for res in data if res.get('protein_segment', '').startswith('TM')}
        return tm_residues if tm_residues else None
    except Exception:
        return None

def load_and_align_structure(pdb_id, chain_id, uniprot_seq, parser=MMCIFParser(QUIET=True)):
    """Loads structure and maps PDB residues to UniProt positions."""
    cif_path = os.path.join(CIF_DIR, f"{pdb_id.lower()}.cif")
    if not os.path.exists(cif_path): return None
    
    try:
        structure = parser.get_structure(pdb_id, cif_path)
        chain = structure[0][chain_id]
        # Filter standard AA only
        pdb_res = [r for r in chain.get_residues() if r.id[0] == ' ' and 'CA' in r]
        pdb_seq = "".join([three_to_one(r.get_resname()) for r in pdb_res])
        
        # Alignment using BLOSUM62 (Corrected for modern BioPython)
        matrix = substitution_matrices.load("BLOSUM62")
        alignments = pairwise2.align.localds(uniprot_seq.replace('-', ''), pdb_seq, matrix, -10, -0.5)
        
        if not alignments: return None
        
        mapping = {} # {UniProt_Pos (1-based) : PDB_Residue_Object}
        p_idx, u_idx = 0, 0
        for u_char, p_char in zip(*alignments[0][:2]):
            if u_char != '-': u_idx += 1
            if p_char != '-':
                if u_char != '-': mapping[u_idx] = pdb_res[p_idx]
                p_idx += 1
        return mapping
    except Exception as e:
        # print(f"Error parsing {pdb_id}: {e}")
        return None

def create_msa_map(uniprot_seq, msa_seq):
    """Maps UniProt positions to MSA column indices."""
    mapping = {}
    u_idx, m_idx = 0, 0
    u_seq_clean = uniprot_seq.replace('-', '')
    while u_idx < len(u_seq_clean) and m_idx < len(msa_seq):
        if msa_seq[m_idx] != '-':
            mapping[u_idx + 1] = m_idx
            u_idx += 1
        m_idx += 1
    return mapping

def get_rep_chain_id(uniprot_id, pdb_id, df_rep_chain):
    """Finds the best chain for a PDB."""
    subset = df_rep_chain[(df_rep_chain['UniProt_ID'] == uniprot_id) & (df_rep_chain['PDB_ID'] == pdb_id)]
    if subset.empty: return None
    return subset.sort_values(by='score', ascending=False).iloc[0]['chain_id']

In [ ]:
# ==============================================================================
# 2. PART 1: MSA DISTANCE vs 3D DISTANCE (Figure 3A)
# ==============================================================================

def run_msa_3d_correlation():
    print("\n--- Running Part 1: MSA vs. 3D Correlation ---")
    
    # Load metadata
    try:
        df_msa = pd.read_csv(MSA_FILE)
        df_rep_chain = pd.read_csv(REP_CHAIN_FILE)
        df_seq = pd.read_csv(SEQUENCE_INFO_FILE)
        df_rep_apo = pd.read_csv(REP_APO_FILE)
    except FileNotFoundError as e:
        print(f"Data missing: {e}. Skipping Part 1.")
        return None
    
    seq_cache = df_seq.set_index('Entry')['Sequence'].to_dict()
    entry_cache = df_seq.set_index('Entry')['Entry Name'].to_dict()
    msa_cache = df_msa.set_index('uniprot_id')['protein_seq'].to_dict()

    # Select Representative Apo Structures (Binding Coverage 100% preferred)
    rep_apo = df_rep_apo[df_rep_apo['Binding_Coverage'] == 100.0].sort_values(['UniProt_ID', 'Resolution'])
    rep_apo_map = rep_apo.drop_duplicates('UniProt_ID').set_index('UniProt_ID')['PDB_ID'].to_dict()

    results = []
    tm_cache = {}

    for uid, pdb in tqdm(rep_apo_map.items(), desc="Processing GPCRs"):
        if uid not in seq_cache or uid not in msa_cache: continue
        
        # Get Chain
        chain_id = get_rep_chain_id(uid, pdb, df_rep_chain)
        if not chain_id: continue

        # Load Structure & Map
        struct_map = load_and_align_structure(pdb, chain_id, seq_cache[uid])
        if not struct_map: continue
        
        msa_map = create_msa_map(seq_cache[uid], msa_cache[uid])
        
        # Filter for TM regions only
        if uid not in tm_cache:
            tm_cache[uid] = get_tm_boundaries(entry_cache[uid])
            time.sleep(0.1) # Rate limit API
        if not tm_cache[uid]: continue
        
        # Common residues in TM region
        common_res = sorted([i for i in struct_map.keys() & msa_map.keys() if i in tm_cache[uid]])
        if len(common_res) < 30: continue # Skip if too few residues

        # Calculate Distance Matrices
        # 1. 3D Euclidean Distance
        coords = np.array([struct_map[i]['CA'].get_coord() for i in common_res])
        d_3d = pdist(coords, 'euclidean')
        
        # 2. MSA Sequence Distance (Column Index Diff)
        msa_idx = np.array([msa_map[i] for i in common_res]).reshape(-1, 1)
        d_msa = pdist(msa_idx, 'cityblock')

        # Correlation
        if np.std(d_3d) > 0 and np.std(d_msa) > 0:
            corr, _ = pearsonr(d_3d, d_msa)
            results.append({'UniProt_ID': uid, 'PDB_ID': pdb, 'correlation': corr})

    df_res = pd.DataFrame(results)
    df_res.to_csv(TM_CORR_RESULTS, index=False)
    print(f"Saved results to {TM_CORR_RESULTS}")
    return df_res

# Run
df_tm_corr = run_msa_3d_correlation()

In [ ]:
# ==============================================================================
# 3. PART 2: CONSERVATION vs STRUCTURAL DYNAMICS (Figure 3B)
# ==============================================================================

def calculate_displacement_and_conservation():
    print("\n--- Running Part 2: Conservation vs. Dynamics ---")
    
    # --- Step A: Calculate Conservation (Shannon Entropy) ---
    try:
        df_msa = pd.read_csv(MSA_FILE)
    except FileNotFoundError:
        print("MSA file missing. Skipping Part 2.")
        return None

    print("Calculating MSA Entropy...")
    msa_matrix = np.array([list(s) for s in df_msa['protein_seq']]).T
    aa_chars = 'ACDEFGHIKLMNPQRSTVWY-'
    
    conservation_scores = []
    for col in tqdm(msa_matrix, desc="MSA Columns"):
        counts = pd.Series(col).value_counts()
        probs = counts / len(col)
        # Normalized Entropy (1 - H/Hmax), 1 = Conserved, 0 = Variable
        score = 1 - (entropy(probs, base=2) / np.log2(len(aa_chars)))
        conservation_scores.append(score)
    
    # --- Step B: Calculate Displacement (Placeholder Logic) ---
    # Note: Full displacement calculation requires superimposing Apo vs Holo structures for ALL pairs.
    # Since the user already has this data (DISP_RESULTS), we will try to load it first.
    # If not found, we skip the heavy calculation to keep the notebook light for Github demo.
    
    if os.path.exists(DISP_RESULTS) and os.path.exists(CONSV_RESULTS):
        print("Loading pre-calculated displacement/conservation files (recommended)...")
        df_disp = pd.read_csv(DISP_RESULTS)
        df_cons = pd.read_csv(CONSV_RESULTS)
    else:
        print("⚠️ Pre-computed displacement file not found.")
        print("Calculating displacements from scratch requires downloading all PDBs.")
        print("Please ensure 'GPCR_Residue_Displacements.csv' is in the output folder.")
        # For the purpose of reproduction script, we stop here if data is missing.
        # The user should upload their processed displacement CSV to data/processed/
        return None

    # --- Step C: Correlate ---
    print("Calculating Correlations...")
    merged = pd.merge(df_disp, df_cons, on=['UniProt_ID', 'UniProt_Position'])
    
    corr_data = []
    for uid, group in merged.groupby('UniProt_ID'):
        if len(group) > 10:
            # Clean NaNs
            group = group.dropna(subset=['Conservation_Score', 'Median_Displacement_A'])
            if len(group) > 10:
                corr, _ = pearsonr(group['Conservation_Score'], group['Median_Displacement_A'])
                corr_data.append({'UniProt_ID': uid, 'pearson_correlation': corr})
            
    df_final_corr = pd.DataFrame(corr_data)
    df_final_corr.to_csv(FINAL_CORR_RESULTS, index=False)
    print(f"Saved correlations to {FINAL_CORR_RESULTS}")
    return df_final_corr

# Run
df_dyn_corr = calculate_displacement_and_conservation()

In [ ]:
# ==============================================================================
# 4. VISUALIZATION (Figure 3 A & B)
# ==============================================================================

def plot_raincloud(data, column, title, filename, color):
    if data is None or data.empty: 
        print(f"No data for {title}. Skipping plot.")
        return
    
    plt.figure(figsize=(6, 4))
    data['Group'] = 'GPCRs' # Dummy group
    
    # Raincloud Plot
    ax = pt.RainCloud(
        data=data, x='Group', y=column,
        palette=[color], width_viol=.8, orient='h',
        move=0.2
    )
    
    # Median Line
    median_val = data[column].median()
    plt.axvline(median_val, color='firebrick', linestyle='--', lw=2)
    plt.text(median_val, -0.45, f'Median = {median_val:.3f}', 
             color='firebrick', ha='center', fontweight='bold')
    
    plt.title(title, fontsize=14, fontweight='bold')
    plt.xlabel("Pearson Correlation Coefficient (PCC)")
    plt.ylabel("")
    plt.yticks([]) # Hide y-axis dummy label
    
    plt.tight_layout()
    save_path = os.path.join(OUTPUT_DIR, filename)
    plt.savefig(save_path, dpi=600)
    print(f"Plot saved to {save_path}")
    plt.show()

# Plot Figure 3A (MSA Distance vs 3D Distance)
if df_tm_corr is not None:
    plot_raincloud(df_tm_corr, 'correlation', 
                   'MSA Distance vs. 3D Cα Distance (Figure 3A)', 
                   'Fig3A_Raincloud.png', '#69b3a2')

# Plot Figure 3B (Conservation vs Dynamics)
if df_dyn_corr is not None:
    plot_raincloud(df_dyn_corr, 'pearson_correlation', 
                   'Conservation vs. Dynamics (Figure 3B)', 
                   'Fig3B_Raincloud.png', '#e76f51')